# Retos 1 y 2 · (borrador de Álvaro)

In [ ]:
from comun import *


df_atletas.shape, df_medallas.shape, df.shape

---
# RETO 1 · Relación entre los datos de entrada
## A) Tareas de análisis

### 1.1 Qué aporta cada tabla y cómo se relacionan
*Explica brevemente (5–8 líneas) qué aporta cada tabla (atletas y medallas) y cómo se relacionan entre sí.*

*Respuesta:*

- **`atletas.csv` (135.571 filas)** describe **quién** es cada deportista: una fila por atleta, con un `ID` único, su nombre, sexo, edad, altura, peso y país (`Team` y `NOC`). Son datos que no cambian de unos Juegos a otros. Por eso cada atleta tiene una sola edad, que no es la que tenía en cada edición.
- **`medallas.csv` (271.116 filas)** describe **qué hizo**: una fila por atleta y prueba disputada, con los Juegos (`Games`, `Year`, `Season`, `City`), el deporte, la prueba y la medalla. Si `Medal` está vacío, el atleta participó pero no ganó medalla, que es lo habitual (231.333 filas, un 85 %).
- **Relación:** se unen por la columna `ID`, con una relación **1 a N**: un atleta aparece una vez en `atletas` y tantas veces como pruebas haya disputado en `medallas`. Hemos comprobado que todos los `ID` de una tabla están en la otra, así que al unirlas no se pierde a nadie.
- **Por qué dos tablas:** si estuviera todo en una, los datos personales se repetirían en cada participación (Phelps tendría su altura repetida en decenas de filas). Separarlas evita esa repetición, pero para casi cualquier análisis interesante (medallas por país, altura de los medallistas, etc.) hay que juntarlas. Eso es lo que hace `df` en `comun.py`.

### 1.2 Listado de ejemplo
*Muestra un listado (5–10 filas) con: nombre del atleta, país, tipo de juegos, años en los que participó y medallas obtenidas en cada año.*

In [ ]:
# Atletas elegidos para el ejemplo: deportistas conocidos de Verano e Invierno,
# con y sin medallas, y uno con un hueco en su carrera (Nadal no fue en 2012).
ids_ejemplo = {
    94406: "Michael Phelps",
    11951: "Ole Einar Bjørndalen",
    67046: "Larysa Latynina",
    9750:  "Mireia Belmonte",
    38960: "Pau Gasol",
    84006: "Rafael Nadal",
}

NOMBRE_MEDALLA = {"Gold": "oro", "Silver": "plata", "Bronze": "bronce"}

def texto_medallas(medallas):
    """Convierte las medallas de un atleta en unos Juegos en un texto: '6 oro, 2 bronce'."""
    conteo = medallas.value_counts()          # value_counts ignora los NaN (sin medalla)
    partes = [f"{conteo[m]} {NOMBRE_MEDALLA[m]}" for m in NOMBRE_MEDALLA if m in conteo]
    return ", ".join(partes) if partes else "sin medalla"

# 1) Filas de los atletas elegidos (tabla unida: necesitamos nombre/país Y medallas).
ejemplo = df[df["ID"].isin(ids_ejemplo)]

# 2) Una fila por atleta y año, con sus medallas de ese año en texto.
#    Aquí se cuenta cada fila: estamos analizando atletas, no el medallero de países.
por_anio = (ejemplo
            .groupby(["ID", "Name", "NOC", "Season", "Year"])["Medal"]
            .apply(texto_medallas)
            .reset_index(name="Medallas"))
por_anio["Detalle"] = por_anio["Year"].astype(str) + ": " + por_anio["Medallas"]

# 3) Una fila por atleta: lista de años y detalle de medallas por año.
listado = (por_anio
           .groupby(["ID", "Name", "NOC", "Season"])
           .agg(Años=("Year", list), Medallas_por_año=("Detalle", "; ".join))
           .reset_index())

# Los nombres del CSV han perdido las tildes (p. ej. "Garca"), así que usamos los nuestros.
listado["Name"] = listado["ID"].map(ids_ejemplo)

listado.drop(columns="ID").rename(columns={
    "Name": "Atleta", "NOC": "País", "Season": "Tipo de Juegos",
    "Medallas_por_año": "Medallas por año"})

### 1.3 Preguntas que requieren las dos tablas
*Indica 3 preguntas que no se podrían responder usando solo una de las tablas.*

1. **¿Qué país ha ganado más medallas de oro en los Juegos de Invierno?** El país (`NOC`) está en `atletas`, y la medalla y la temporada están en `medallas`.
2. **¿Las nadadoras que ganan medalla son más altas que las que no la ganan?** La altura y el sexo están en `atletas`, y el deporte y la medalla en `medallas`.
3. **¿Cómo ha cambiado el porcentaje de mujeres participantes década a década?** El sexo está en `atletas` y el año de los Juegos en `medallas`.

### 1.4 Participaciones por país (Top-10, Verano/Invierno)
*Gráfico de barras apiladas con el nº de participaciones por país para los 10 países con más participaciones, separando Verano/Invierno. Justifica la elección del gráfico.*

In [ ]:
# Participación = un atleta en unos Juegos. Quitamos duplicados por (ID, Games)
# para no contar varias veces a quien compite en varias pruebas en la misma edición.
participaciones = (df
                   .drop_duplicates(subset=["ID", "Games"])
                   .groupby(["NOC", "Season"])
                   .size()
                   .unstack(fill_value=0))       # columnas: Summer, Winter

# Top-10 por total (Verano + Invierno), ya ordenado de mayor a menor.
top10 = participaciones.loc[participaciones.sum(axis=1).nlargest(10).index]

ax = (top10
      .rename(columns={"Summer": "Verano", "Winter": "Invierno"})
      .plot(kind="bar", stacked=True,
            color=[COLORES_TEMPORADA["Summer"], COLORES_TEMPORADA["Winter"]]))
ax.set_title("Participaciones por país (Top-10), Verano e Invierno")
ax.set_xlabel("País (NOC)")
ax.set_ylabel("Nº de participaciones (atleta × Juegos)")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Temporada")
plt.show()

# Tabla de apoyo: % de participaciones de Invierno en cada país.
top10.assign(Pct_Invierno=(top10["Winter"] / top10.sum(axis=1) * 100).round(1))

*Justificación:* comparamos una cantidad entre categorías (países), así que lo natural es un gráfico de **barras**. Al **apilarlas** se ven dos cosas a la vez: la altura total de la barra da el número total de participaciones del país (por eso las barras van ordenadas de mayor a menor), y los dos tramos muestran cuánto aporta cada temporada.

*Qué se ve:*
- **EE. UU.** destaca muy por encima del resto, con unas 12.800 participaciones.
- En todos los países **domina el Verano**, porque hay muchas más pruebas y deportistas. El peso del Invierno varía mucho: es alto en **Canadá (28 %)** y **Suecia (26 %)**, y muy bajo en **Australia (6 %)**, lo que encaja con el clima de cada país.
- **Limitación:** algunos países aparecen partidos en varios códigos. Alemania tiene `GER`, `FRG` y `GDR`, y la Unión Soviética (`URS`) y Rusia (`RUS`) van por separado. Por eso Alemania y Rusia salen por debajo de lo que les correspondería históricamente.

## B) Función `resumen_participacion(id_atleta, df_medallas)`
*Resume los años de participación de un atleta en intervalos consecutivos.*

- **Entrada:** `id_atleta` (int), `df_medallas` (DataFrame)
- **Salida:** `{"participaciones": int, "intervalos": ["AAAA-AAAA", ...]}`
- **Criterio de consecutivo:** dos participaciones son consecutivas si corresponden a **ediciones seguidas de la misma temporada** (Verano o Invierno), no si se llevan 4 años. Para ello se numeran las ediciones que realmente se celebraron según los propios datos y se compara su posición. Así:
  - Invierno 1992 → 1994 cuenta como seguido (cambio de calendario).
  - Verano 1904 → 1906 → 1908 cuenta como seguido (los Juegos de 1906 están en los datos).
  - Verano 1936 → 1948 cuenta como seguido (no hubo Juegos en 1940 ni 1944).
- **Otras decisiones:**
  - Las participaciones se cuentan como `Games` distintos, no como filas.
  - Las rachas de Verano y de Invierno se calculan por separado y la lista se ordena por año de inicio.
  - Una participación aislada se escribe `"AAAA-AAAA"` (por ejemplo `"2008-2008"`), para que todos los intervalos tengan el mismo formato.
  - Si el `ID` no existe, se devuelve `{"participaciones": 0, "intervalos": []}`.

In [ ]:
def resumen_participacion(id_atleta, df_medallas):
    """
    Resume en qué Juegos ha participado un atleta, agrupando los años en
    intervalos de ediciones consecutivas.

    Parámetros
    ----------
    id_atleta : int
        ID del atleta (columna `ID`).
    df_medallas : DataFrame
        Tabla de participaciones (una fila por atleta y prueba).

    Devuelve
    --------
    dict
        {"participaciones": nº de Juegos distintos,
         "intervalos": lista de textos "AAAA-AAAA", ordenada por año de inicio}

    Ejemplo
    -------
    >>> resumen_participacion(94406, df_medallas)   # Michael Phelps
    {'participaciones': 5, 'intervalos': ['2000-2016']}
    """
    # 1) Nos quedamos solo con las filas de este atleta (máscara booleana).
    filas = df_medallas[df_medallas["ID"] == id_atleta]

    # Si el ID no aparece, no ha participado en nada.
    if filas.empty:
        return {"participaciones": 0, "intervalos": []}

    # 2) Participaciones = Juegos distintos, NO filas.
    #    Phelps tiene muchas filas por edición (una por prueba), pero son 5 Juegos.
    participaciones = filas["Games"].nunique()

    intervalos = []

    # 3) Verano e Invierno tienen calendarios distintos, así que cada
    #    temporada se trata por separado.
    for temporada in filas["Season"].unique():

        # 3a) Todas las ediciones que se celebraron en esta temporada,
        #     ordenadas. Se sacan de los propios datos, así incluyen 1906
        #     y excluyen 1940/1944 sin tener que escribirlo a mano.
        ediciones = sorted(
            df_medallas.loc[df_medallas["Season"] == temporada, "Year"].unique()
        )

        # 3b) Diccionario año -> posición de la edición: {1896: 0, 1900: 1, ...}
        #     Trabajar con posiciones en lugar de años es la clave:
        #     dos ediciones son seguidas si sus posiciones se diferencian en 1.
        posicion = {anio: i for i, anio in enumerate(ediciones)}

        # 3c) Años (sin repetir y ordenados) en los que compitió el atleta.
        anios = sorted(filas.loc[filas["Season"] == temporada, "Year"].unique())

        # 3d) Recorremos los años partiéndolos en rachas.
        #     `inicio`  = primer año de la racha actual
        #     `anterior` = último año añadido a la racha actual
        inicio = anterior = anios[0]
        for anio in anios[1:]:
            if posicion[anio] == posicion[anterior] + 1:
                # Es la edición siguiente: la racha continúa.
                anterior = anio
            else:
                # Hay un hueco: cerramos la racha y empezamos otra.
                intervalos.append(f"{inicio}-{anterior}")
                inicio = anio
                anterior = anio
        # La última racha se queda abierta al salir del bucle: hay que cerrarla.
        intervalos.append(f"{inicio}-{anterior}")

    # 4) Ordenamos por año de inicio. Como todos empiezan por un año de 4 cifras,
    #    el orden alfabético de los textos coincide con el cronológico.
    intervalos.sort()

    return {"participaciones": participaciones, "intervalos": intervalos}

In [ ]:
# Caso de prueba (Michael Phelps)
assert resumen_participacion(94406, df_medallas) == {"participaciones": 5, "intervalos": ["2000-2016"]}
resumen_participacion(94406, df_medallas)

---
# RETO 2 · Perfil de las/los atletas
## A) Tareas de análisis

### 2.1 Edad media Verano vs Invierno
*Obtén la edad media diferenciando entre juegos de Verano e Invierno y compáralas en 3–5 líneas.*

In [ ]:
# TODO


*Comparación:* TODO

### 2.2 Histograma de edades de los 5 países con más atletas
*FacetGrid de histogramas por país.*

In [ ]:
# TODO


### 2.3 Distribución porcentual Hombre/Mujer × Verano/Invierno
*En porcentajes, no conteos. Justifica el tipo de gráfica.*

In [ ]:
# TODO


*Justificación:* TODO

### 2.4 Altura y peso de las mujeres en los 5 deportes con más mujeres que hombres
*Calcula medias y muestra una visualización comparativa (p. ej., boxplots). Al menos 2 comentarios.*

In [ ]:
# TODO


*Comentarios:*
1. TODO
2. TODO

## B) Función `extremos_atletas(df_atletas)`
*Devuelve los atletas extremos (más joven/mayor, alto/bajo, ligero/pesado).*

**Umbrales de plausibilidad (justificar):**
| Variable | Mín | Máx | Motivo |
|---|---|---|---|
| Edad | TODO | TODO | TODO |
| Altura (cm) | TODO | TODO | TODO |
| Peso (kg) | TODO | TODO | TODO |

**Desempate:** más participaciones → orden alfabético del nombre.

In [ ]:
def extremos_atletas(df_atletas):
    # TODO (usa resumen_participacion para desempatar)
    pass

In [ ]:
extremos_atletas(df_atletas)